# TV8 – NỘI DUNG 5B: ĐÁNH GIÁ & DỰ BÁO (R / Google Colab)

**Mục tiêu:** tính MAE, RMSE, MASE; so sánh các mô hình (Mean, Naive, Seasonal Naive, ARIMA, SARIMA); chọn mô hình tối ưu; xuất dự báo **14 ngày** tiếp theo; viết kết luận & hạn chế.

**Cách chạy:** Runtime → Change runtime type → **R**. Upload `daily_orders.csv` vào mục Files rồi chạy lần lượt các cell.

**Sản phẩm sinh ra:** `model_comparison.csv`, `model_selection.csv`, `forecast.csv`, `plot_test_comparison.png`, `plot_metrics.png`, `plot_forecast_14d.png`

In [ ]:
# ---------- 0. PACKAGES ----------
pkgs <- c("readr","dplyr","tidyr","ggplot2","forecast","tseries","lubridate")
for (p in pkgs) {
  if (!require(p, character.only = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org", quiet = TRUE)
  }
  library(p, character.only = TRUE)
}
set.seed(123)
cat("Đã load xong packages\n")

In [ ]:
# ---------- 1. ĐỌC DỮ LIỆU (daily_orders.csv: date, total_orders) ----------
paths <- c("daily_orders.csv", "/content/daily_orders.csv")
csv_path <- paths[file.exists(paths)][1]
if (is.na(csv_path)) stop("Không tìm thấy daily_orders.csv - hãy upload file vào Colab!")

daily <- read_csv(csv_path, show_col_types = FALSE)
names(daily)[1:2] <- c("date", "orders")
daily <- daily %>%
  mutate(date = as.Date(date)) %>%
  arrange(date) %>%
  complete(date = seq(min(date), max(date), by = "day"), fill = list(orders = 0))

FREQ <- 7     # chu kỳ tuần
H    <- 14    # horizon dự báo & kích thước tập test
y    <- daily$orders
n    <- length(y)

cat("Khoảng thời gian:", format(min(daily$date)), "->", format(max(daily$date)), "\n")
cat("Số ngày:", n, "| TB đơn/ngày:", round(mean(y), 2), "| SD:", round(sd(y), 2), "\n")

In [ ]:
# ---------- 2. CHIA TRAIN / TEST (14 ngày cuối làm test) ----------
n_train  <- n - H
ts_all   <- ts(y, frequency = FREQ)
ts_train <- ts(y[1:n_train], frequency = FREQ)
test     <- y[(n_train + 1):n]
test_dates <- daily$date[(n_train + 1):n]

cat("Train:", length(ts_train), "ngày | Test:", length(test), "ngày\n")
cat("Test từ", format(test_dates[1]), "đến", format(test_dates[H]), "\n")

## 1. Huấn luyện các mô hình trên tập train
- **Mean, Naive, Seasonal Naive:** mô hình cơ sở (baseline) do TV6 đề xuất.
- **ARIMA (không mùa vụ):** `auto.arima(seasonal = FALSE, d = 0)` – *phần code còn thiếu của nhóm, bổ sung ở đây*. d = 0 theo kết luận ADF của TV6.
- **SARIMA:** `auto.arima(seasonal = TRUE, d = 0, D = 0)` – giống cấu hình trong `model_advanced.ipynb`.

In [ ]:
# ---------- 3. FIT CÁC MÔ HÌNH ----------
fit_mean   <- meanf(ts_train,  h = H, level = c(80, 95))
fit_naive  <- naive(ts_train,  h = H, level = c(80, 95))
fit_snaive <- snaive(ts_train, h = H, level = c(80, 95))

fit_arima <- auto.arima(ts_train, seasonal = FALSE, d = 0,
                        stepwise = FALSE, approximation = FALSE)

fit_sarima <- auto.arima(ts_train, seasonal = TRUE, d = 0, D = 0,
                         stepwise = FALSE, approximation = FALSE)

fc_arima  <- forecast(fit_arima,  h = H, level = c(80, 95))
fc_sarima <- forecast(fit_sarima, h = H, level = c(80, 95))

# Nhãn mô hình (p,d,q)(P,D,Q)[m]
spec_arima  <- function(f) { a <- f$arma; sprintf("ARIMA(%d,%d,%d)", a[1], a[6], a[2]) }
spec_sarima <- function(f) { a <- f$arma; sprintf("SARIMA(%d,%d,%d)(%d,%d,%d)[%d]", a[1], a[6], a[2], a[3], a[7], a[4], a[5]) }

cat("ARIMA  chọn được :", spec_arima(fit_arima),  "\n")
cat("SARIMA chọn được :", spec_sarima(fit_sarima), "\n\n")
summary(fit_arima)
summary(fit_sarima)

## 2. Chỉ số sai số: MAE, RMSE, MAPE, MASE
- MAE = trung bình |sai số|; RMSE = căn trung bình bình phương sai số.
- **MASE** = MAE(test) / MAE của *seasonal naive trong mẫu train* (lag = 7). Cách tính này trùng với `accuracy()` của TV6 (Naive 0.8634, Seasonal Naive 1.1482). MASE < 1 nghĩa là tốt hơn seasonal naive trên train.

In [ ]:
# ---------- 4. TÍNH CHỈ SỐ SAI SỐ ----------
scale_mase <- mean(abs(diff(as.numeric(ts_train), lag = FREQ)))

calc_metrics <- function(actual, pred, name, spec) {
  e <- actual - pred
  data.frame(Model = name, Spec = spec,
             MAE  = mean(abs(e)),
             RMSE = sqrt(mean(e^2)),
             MAPE = mean(abs(e / actual)) * 100,
             MASE = mean(abs(e)) / scale_mase)
}

preds <- list(
  "Mean"           = as.numeric(fit_mean$mean),
  "Naive"          = as.numeric(fit_naive$mean),
  "Seasonal Naive" = as.numeric(fit_snaive$mean),
  "ARIMA"          = as.numeric(fc_arima$mean),
  "SARIMA"         = as.numeric(fc_sarima$mean)
)
specs <- c("Mean (trung bình train)", "Naive", "Seasonal Naive [7]",
           spec_arima(fit_arima), spec_sarima(fit_sarima))

results <- bind_rows(lapply(seq_along(preds), function(i)
  calc_metrics(test, preds[[i]], names(preds)[i], specs[i])))

# Kiểm định Ljung-Box trên phần dư (lag = 10) - theo hàm chẩn đoán 5A
fits_list <- list(fit_mean, fit_naive, fit_snaive, fit_arima, fit_sarima)
results$LjungBox_p <- sapply(fits_list, function(f) {
  r <- na.omit(as.numeric(residuals(f)))
  Box.test(r, lag = 10, type = "Ljung-Box")$p.value
})
results$Residual_OK <- ifelse(results$LjungBox_p > 0.05, "Có (nhiễu trắng)", "Không")
results$simplicity  <- seq_len(nrow(results))   # độ đơn giản (nhỏ = đơn giản hơn)

comparison <- results %>% select(Model, Spec, MAE, RMSE, MAPE, MASE, LjungBox_p, Residual_OK)
comparison_print <- comparison %>% mutate(across(where(is.numeric), ~ round(.x, 4)))

cat("===== BẢNG SO SÁNH MAE / RMSE / MAPE / MASE (tập test 14 ngày) =====\n")
print(comparison_print, row.names = FALSE)
write_csv(comparison_print, "model_comparison.csv")

## 3. Chọn mô hình tối ưu
**Quy tắc chọn:** (1) MASE thấp nhất; (2) nếu bằng nhau thì RMSE thấp nhất; (3) nếu vẫn bằng nhau thì chọn mô hình **đơn giản hơn** (nguyên tắc tiết kiệm tham số). Đồng thời kiểm tra phần dư có là nhiễu trắng (Ljung-Box p > 0.05).

In [ ]:
# ---------- 5. BẢNG CHỌN MODEL TỐI ƯU ----------
ord <- order(round(results$MASE, 6), round(results$RMSE, 6), results$simplicity)
sel <- results[ord, ]
sel$Rank     <- seq_len(nrow(sel))
sel$Selected <- ifelse(sel$Rank == 1, "CHỌN", "")
best_name    <- sel$Model[1]
best_row     <- sel[1, ]

selection <- sel %>%
  select(Rank, Model, Spec, MAE, RMSE, MAPE, MASE, LjungBox_p, Residual_OK, Selected) %>%
  mutate(across(where(is.numeric), ~ round(.x, 4)))

cat("===== BẢNG CHỌN MÔ HÌNH TỐI ƯU =====\n")
print(selection, row.names = FALSE)
write_csv(selection, "model_selection.csv")

cat("\n=> Mô hình tối ưu:", best_name, "-", best_row$Spec, "\n")
ties <- sel$Model[abs(sel$MASE - best_row$MASE) < 1e-6 & abs(sel$RMSE - best_row$RMSE) < 1e-6]
if (length(ties) > 1) {
  cat("   Lưu ý: các mô hình sau cho kết quả TRÙNG NHAU trên tập test:", paste(ties, collapse = ", "), "\n")
  cat("   (ARIMA/SARIMA tự chọn bậc (0,0,0) có hằng số = mô hình Mean) -> chọn mô hình đơn giản nhất.\n")
}

In [ ]:
# ---------- 6. BIỂU ĐỒ SO SÁNH ----------
# 6a. Dự báo từng mô hình vs thực tế trên tập test
hist_df <- data.frame(date = daily$date[(n_train - 29):n_train], value = y[(n_train - 29):n_train])
cmp_df <- bind_rows(
  lapply(names(preds), function(nm) data.frame(date = test_dates, value = preds[[nm]], series = nm)),
  data.frame(date = test_dates, value = test, series = "Thực tế")
)
cmp_df$series <- factor(cmp_df$series, levels = c("Thực tế", names(preds)))

p_cmp <- ggplot() +
  geom_line(data = hist_df, aes(date, value), colour = "grey60") +
  geom_line(data = cmp_df, aes(date, value, colour = series, linetype = series,
                               linewidth = series == "Thực tế")) +
  scale_linewidth_manual(values = c(`TRUE` = 1.3, `FALSE` = 0.8), guide = "none") +
  labs(title = "So sánh dự báo các mô hình trên tập test (14 ngày)",
       subtitle = "Mean, ARIMA(0,0,0) và SARIMA(0,0,0) có thể chồng khít nhau",
       x = "Ngày", y = "Số đơn / ngày", colour = "Mô hình", linetype = "Mô hình") +
  theme_minimal(base_size = 13) + theme(legend.position = "bottom")
print(p_cmp)
ggsave("plot_test_comparison.png", p_cmp, width = 10, height = 5, dpi = 150)

# 6b. Cột so sánh MAE / RMSE / MASE
metric_long <- selection %>%
  select(Model, Selected, MAE, RMSE, MASE) %>%
  pivot_longer(c(MAE, RMSE, MASE), names_to = "Metric", values_to = "Value")
metric_long$Model <- factor(metric_long$Model, levels = unique(results$Model))

p_met <- ggplot(metric_long, aes(Model, Value, fill = Model)) +
  geom_col(show.legend = FALSE) +
  geom_text(aes(label = round(Value, 2)), vjust = -0.3, size = 3.3) +
  facet_wrap(~ Metric, scales = "free_y") +
  labs(title = "So sánh sai số các mô hình (tập test 14 ngày)", x = NULL, y = NULL) +
  theme_minimal(base_size = 12) + theme(axis.text.x = element_text(angle = 30, hjust = 1))
print(p_met)
ggsave("plot_metrics.png", p_met, width = 11, height = 4.5, dpi = 150)

## 4. (Bổ sung) Kiểm tra độ ổn định bằng rolling-origin (5 lần, mỗi lần dự báo 14 ngày)
Một tập test 14 ngày duy nhất khá "nhiễu", nên cell này kiểm tra xem thứ hạng mô hình có ổn định không. (Có thể bỏ qua nếu muốn chạy nhanh.)

In [ ]:
# ---------- 7. ROLLING-ORIGIN CHECK ----------
origins <- n - H * (5:1)
cv_list <- list()
for (o in origins) {
  tr <- ts(y[1:o], frequency = FREQ)
  te <- y[(o + 1):(o + H)]
  sc <- mean(abs(diff(as.numeric(tr), lag = FREQ)))
  f <- list(
    "Mean"           = rep(mean(y[1:o]), H),
    "Naive"          = rep(y[o], H),
    "Seasonal Naive" = as.numeric(snaive(tr, h = H)$mean),
    "ARIMA"          = as.numeric(forecast(auto.arima(tr, seasonal = FALSE, d = 0), h = H)$mean),
    "SARIMA"         = as.numeric(forecast(auto.arima(tr, seasonal = TRUE, d = 0, D = 0), h = H)$mean)
  )
  for (nm in names(f)) {
    e <- te - f[[nm]]
    cv_list[[length(cv_list) + 1]] <- data.frame(
      origin = o, Model = nm,
      MAE = mean(abs(e)), RMSE = sqrt(mean(e^2)), MASE = mean(abs(e)) / sc)
  }
}
cv_summary <- bind_rows(cv_list) %>%
  group_by(Model) %>%
  summarise(MAE = mean(MAE), RMSE = mean(RMSE), MASE = mean(MASE), .groups = "drop") %>%
  arrange(MASE) %>%
  mutate(across(where(is.numeric), ~ round(.x, 4)))

cat("===== ROLLING-ORIGIN (trung bình 5 lần x 14 ngày) =====\n")
print(cv_summary, row.names = FALSE)
write_csv(cv_summary, "rolling_origin_summary.csv")

## 5. Dự báo 14 ngày tiếp theo bằng mô hình tối ưu
Mô hình tối ưu được **fit lại trên toàn bộ 180 ngày** rồi dự báo 14 ngày kế tiếp (30/04/2025 → 13/05/2025).

In [ ]:
# ---------- 8. REFIT TOÀN BỘ DỮ LIỆU & DỰ BÁO 14 NGÀY ----------
refit_arima <- function(fit, series) {
  a <- fit$arma
  Arima(series, order = c(a[1], a[6], a[2]),
        seasonal = list(order = c(a[3], a[7], a[4]), period = a[5]),
        include.mean = any(names(coef(fit)) %in% c("intercept", "mean")))
}

fc_final <- switch(best_name,
  "Mean"           = meanf(ts_all,  h = H, level = c(80, 95)),
  "Naive"          = naive(ts_all,  h = H, level = c(80, 95)),
  "Seasonal Naive" = snaive(ts_all, h = H, level = c(80, 95)),
  "ARIMA"          = forecast(refit_arima(fit_arima,  ts_all), h = H, level = c(80, 95)),
  "SARIMA"         = forecast(refit_arima(fit_sarima, ts_all), h = H, level = c(80, 95))
)

future_dates <- seq(max(daily$date) + 1, by = "day", length.out = H)

forecast_df <- data.frame(
  date         = future_dates,
  weekday      = weekdays(future_dates),
  forecast     = round(as.numeric(fc_final$mean), 2),
  forecast_int = round(as.numeric(fc_final$mean)),
  lower80      = round(as.numeric(fc_final$lower[, 1]), 2),
  upper80      = round(as.numeric(fc_final$upper[, 1]), 2),
  lower95      = round(as.numeric(fc_final$lower[, 2]), 2),
  upper95      = round(as.numeric(fc_final$upper[, 2]), 2),
  model        = paste(best_name, "-", best_row$Spec)
)
write_csv(forecast_df, "forecast.csv")

cat("===== DỰ BÁO 14 NGÀY TIẾP THEO (", best_name, ") =====\n")
print(forecast_df, row.names = FALSE)

In [ ]:
# ---------- 9. BIỂU ĐỒ DỰ BÁO 14 NGÀY ----------
hist60 <- tail(daily, 60)
p_fc <- ggplot() +
  geom_line(data = hist60, aes(date, orders, colour = "Thực tế (60 ngày gần nhất)"), linewidth = 0.9) +
  geom_ribbon(data = forecast_df, aes(date, ymin = lower95, ymax = upper95, fill = "KTC 95%"), alpha = 0.18) +
  geom_ribbon(data = forecast_df, aes(date, ymin = lower80, ymax = upper80, fill = "KTC 80%"), alpha = 0.30) +
  geom_line(data = forecast_df, aes(date, forecast, colour = "Dự báo 14 ngày"), linewidth = 1.2, linetype = "dashed") +
  geom_point(data = forecast_df, aes(date, forecast, colour = "Dự báo 14 ngày"), size = 1.8) +
  scale_colour_manual(values = c("Thực tế (60 ngày gần nhất)" = "#1f77b4", "Dự báo 14 ngày" = "#d62728")) +
  scale_fill_manual(values = c("KTC 95%" = "#d62728", "KTC 80%" = "#d62728")) +
  labs(title = "Dự báo số đơn hàng 14 ngày tiếp theo",
       subtitle = paste("Mô hình:", best_name, "-", best_row$Spec),
       x = "Ngày", y = "Số đơn / ngày", colour = "", fill = "") +
  theme_minimal(base_size = 13) + theme(legend.position = "bottom")
print(p_fc)
ggsave("plot_forecast_14d.png", p_fc, width = 10, height = 5, dpi = 150)

## 6. Kết luận & hạn chế của đề tài

In [ ]:
# ---------- 10. KẾT LUẬN TỰ ĐỘNG (số liệu lấy từ kết quả chạy) ----------
b  <- best_row
bs <- results[results$Model == "Seasonal Naive", ]
cat("================ KẾT LUẬN ================\n")
cat(sprintf("1. Chuỗi %d ngày (%s -> %s), TB %.1f đơn/ngày, ADF cho thấy chuỗi dừng (d = 0, D = 0).\n",
            n, format(min(daily$date)), format(max(daily$date)), mean(y)))
cat(sprintf("2. Mô hình tối ưu: %s (%s) - MAE = %.2f, RMSE = %.2f, MAPE = %.2f%%, MASE = %.3f.\n",
            b$Model, b$Spec, b$MAE, b$RMSE, b$MAPE, b$MASE))
cat(sprintf("3. So với Seasonal Naive (MAE = %.2f, RMSE = %.2f, MASE = %.3f): MAE giảm %.1f%%.\n",
            bs$MAE, bs$RMSE, bs$MASE, (1 - b$MAE / bs$MAE) * 100))
cat(sprintf("4. Dự báo 14 ngày (%s -> %s): trung bình %.1f đơn/ngày, KTC 95%% khoảng [%.0f ; %.0f].\n",
            format(min(future_dates)), format(max(future_dates)),
            mean(forecast_df$forecast), min(forecast_df$lower95), max(forecast_df$upper95)))
cat("==========================================\n")

### Kết luận (viết sẵn – đối chiếu lại số khi chạy xong)
1. **Dữ liệu:** 180 ngày đơn hàng (01/11/2024 → 29/04/2025), trung bình ≈ 55.6 đơn/ngày, độ lệch chuẩn ≈ 7.3. ADF p ≈ 0.01 → chuỗi **dừng**, `d = 0`, `D = 0`; ACF/PACF không cho thấy tự tương quan hay mùa vụ tuần đáng kể.
2. **So sánh mô hình (test 14 ngày cuối):**

| Mô hình | MAE | RMSE | MAPE (%) | MASE |
|---|---|---|---|---|
| Mean | 6.531 | 7.680 | 11.39 | 0.814 |
| Naive | 6.929 | 8.648 | 13.30 | 0.863 |
| Seasonal Naive | 9.214 | 11.003 | 16.28 | 1.148 |
| ARIMA (auto, d=0) | ≈ 6.531 | ≈ 7.680 | ≈ 11.39 | ≈ 0.814 |
| SARIMA (auto, d=0, D=0) | 6.531 | 7.680 | 11.39 | 0.814 |

   *(Các dòng ARIMA lấy theo kỳ vọng: `auto.arima` chọn (0,0,0) có hằng số giống SARIMA nên trùng Mean – hãy đối chiếu với bảng thực tế sau khi chạy.)*
3. **Mô hình tối ưu:** ARIMA/SARIMA bậc (0,0,0) có hằng số **trùng hoàn toàn** với mô hình Mean (dự báo phẳng ≈ trung bình). Theo nguyên tắc tiết kiệm tham số, chọn mô hình Mean ≡ ARIMA(0,0,0); MASE = 0.81 < 1 nên tốt hơn Seasonal Naive; Seasonal Naive tệ nhất (MASE > 1) → **không có mùa vụ tuần đủ mạnh**.
4. **Dự báo 14 ngày (30/04 → 13/05/2025):** ≈ 55–56 đơn/ngày, KTC 95% khoảng 41–70 đơn/ngày.

### Hạn chế của đề tài
- Chuỗi gần như **nhiễu trắng quanh mức trung bình** → các mô hình ARIMA/SARIMA không tìm được cấu trúc để khai thác; dự báo là một đường phẳng, không dự báo được biến động từng ngày.
- Chỉ có **180 ngày (~6 tháng)**: chưa đủ để nhận ra mùa vụ năm, ngày lễ, xu hướng dài hạn.
- Đánh giá trên **một tập test 14 ngày** nên chênh lệch giữa Mean và Naive có thể do ngẫu nhiên (đã kiểm tra thêm bằng rolling-origin).
- Chưa đưa **biến ngoại sinh** (khuyến mãi, thời tiết, ngày lễ, thành phố/nhà hàng) và chưa xử lý đơn bị hủy riêng.
- Số đơn là **dữ liệu đếm** (số nguyên) – có thể thử mô hình Poisson/Negative Binomial; chưa thử ETS, Prophet, mô hình học máy.
- Khoảng tin cậy giả định phần dư chuẩn, phương sai không đổi.